#### 순차 데이터
- 텍스트 데이터 : ex) I am a boy , 순서가 의미가 있다.
- 시계열 데이터 : ex) 주식

#### 순환 신경망(RNN : Recurrent Neural Network)

---- 
#### IMDB
- IMDB(Internet Movie Data Base)
- IMDB 구성 : Train Data(25000개 중 긍정 12500, 부정 12500), Test Data(25000개 중 긍정 12500, 부정 12500)
- NLP : Natural Language Process(자연어 처리)
- 말뭉치 : 하나의 데이터셋
- 토큰 : 하나의 단어를 의미
- 어휘사전 : 번호로 구분된 유일한 단어들의 집합

#### Keras로 IMDB 불러오기
: IMDB는 어휘사전으로 변환 되어 있음

In [ ]:
from tensorflow.keras.datasets import imdb

(train_input, train_target),(test_input, test_target) = \
imdb.load_data(num_words=500) # 500개의 단어만 사용

In [ ]:
# Train과 Test의 크기 확인
print(len(train_input), len(test_input))

In [ ]:
# Train의 첫번째 문장의 token 갯수
len(train_input[0])

In [ ]:
# train의 두번째 문장의 Token 갯수
len(train_input[1])

> 영화에 대한 댓글의 길이가 다르다

In [ ]:
# 첫번째 댓글의 출력
print(train_input[0])

- 모든 샘플의 시작 부분의 토큰은 1을 
- 2는 선정한 단어 갯수에 포함되지 않는 단어.
- train_input의 자체는 numpy배열이나 사용자마다 "댓글에 사용한 토큰수가 다르기 때문에 numpy배열을 사용하지 못하고 python의 list를 사용했다.

In [ ]:
# train의 target 출력
print(train_target[:10])

----
#### 훈련세트 준비

In [ ]:
from sklearn.model_selection import train_test_split

train_input,val_input, train_target,  val_target = \
   train_test_split(
      train_input,
      train_target,
      test_size=0.2,
      random_state=42
)

In [ ]:
print(train_input)
print(val_input)
print(train_target)
print(val_target)

----
#### 각 리뷰마다 문자길이 시각화

In [ ]:
import numpy as np

In [ ]:
trainLength = []
for i in train_input:
   trainLength.append(len(i))


np.array(trainLength)

In [ ]:
# List Comprehension
totalLength = np.array(len(i) for i in train_input)
totalLength

In [ ]:
# 0부터 9까비의 숫자 중 짝수만 List에 추가하기
# data : range(10)

numList = [i for i in range(10) if i%2 == 0]
numList

In [ ]:
# 평균과 중앙값
print(np.mean(trainLength), np.median(trainLength))

> 중앙값보다 평균이 크므로 예상치 않게 길게 달린 댓글이 있다고 예측 할 수 있다.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(trainLength)
plt.xlabel('Length')
plt.ylabel('Frequency')
plt.show()

----
#### Sequence Padding
: 전체 자릿수를 100으로 가정했을 경우 한 문장에 토큰이 3개 있을 경우 97개는 0으로 채운다.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_seq = pad_sequences(train_input, maxlen=100)
val_seq = pad_sequences(val_input, maxlen=100)

In [ ]:
# 크기 확인
print(train_seq.shape, val_seq.shape)

In [ ]:
# 첫번째 댓글 확인
print(train_seq[0])

----
#### 순환 신경망 모델 만들기

In [ ]:
train_seq.shape

In [ ]:
# One-hot Encoding
from tensorflow import keras

train_oh = keras.utils.to_categorical(train_seq)
print(train_oh.shape)

In [ ]:
train_oh[1]

In [ ]:
from tensorflow.keras.layers import Dense

In [ ]:
model = keras.Sequential()
model.add(
   keras.layers.SimpleRNN(
      8,
      input_shape = (100,500)
   )
)

model.add(
      Dense(
      1,
      activation='sigmoid'
   )
)

In [ ]:
# 검증 세트도 one hot encoding
val_oh = keras.utils.to_categorical(val_seq)

In [ ]:
model.summary()

In [ ]:
# Loss & Leraning Rate
model.compile(
   
   optimizer =  'rmsprop',
   loss = 'binary_crossentropy',
   metrics = ['accuracy']
)

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint("../Data/best_simplernn.keras")

early_stopping_cb = keras.callbacks.EarlyStopping(
   patience=3,
   restore_best_weights=True
)

In [ ]:
history = model.fit(
   train_oh,
   train_target,
   epochs=100,
   batch_size=64,
   validation_data=(val_oh, val_target),
   callbacks=[checkpoint_cb, early_stopping_cb]
)

> 전체 댓글의 토큰 중 100개만 학습하였더니 80% 이상의 예측력으로 보얐다.

In [ ]:
# 시각화 해보기
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
# plt.legend([''])

plt.show()

In [ ]:
# 평가하기
model.evaluate(train_oh, train_target)

In [ ]:
# 평가하기
model.evaluate(val_oh, val_target)

----
#### LSTM (Long Short Term Memory)

In [ ]:
model = keras.Sequential()
model.add(
   keras.layers.Embedding(500, 16)
)

model.add(
   keras.layers.LSTM(8)
)
model.add(
   keras.layers.Dense(1, activation='sigmoid')
)

model.build(
   input_shape=(None,100)
)

In [ ]:
model.compile(
   optimizer='rmsprop',
   loss = 'binary_crossentropy',
   metrics=['accuracy']
)

In [ ]:
checkpoint_cb = keras.callbacks.ModelCheckpoint("../Data/best_lstm.keras")

early_stopping_cb = keras.callbacks.EarlyStopping(
   patience=3,
   restore_best_weights=True
)

In [ ]:
history = model.fit(
   train_oh,
   train_target,
   epochs=100,
   batch_size=64,
   validation_data=(val_seq, val_target),
   callbacks=[checkpoint_cb, early_stopping_cb]
)

In [ ]:
model.evaluate(val_seq, val_target)

In [ ]:
# 시각화 해보기
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
# plt.legend([''])

plt.show()